# 02 · Báo cáo 1 (Slides 8–12): Khám phá dữ liệu Giao dịch (`TRX`)

Đối chiếu trực tiếp với **Báo cáo 1 (`MBD MINI UNDERSTANDING DATA.pdf`, Slides 8–12)**:
- **Slide 8 (Chất lượng & Currency TRX):** 99,8% đầy đủ, `currency` mã 11 chiếm 99,895% (13 mã), 1,5% (`1.503` khách) không có log TRX, các cặp thuộc tính null cùng nhau (`src_type11/12`, `dst_type11/12`, `src_type21/22`, `src_type31/32`).
- **Slide 9 (Phân vị giao dịch & Amount):** Boxplot số GD/khách (P25=38, P50=164, P75=503) và tổng amount mã 11 (P25=2,65M, P50=17,69M, P75=79,45M); ngoại lai lớn nhất `1.370.187.366.400`.
- **Slide 10 (Mức độ tập trung - Lorenz):** Top 1% khách tạo `13,16%` GD; Top 5% tạo `30,66%`.
- **Slide 11 (RFM: Mua và Không mua):** Recency (số ngày từ GD gần nhất), Frequency (`T-1`), Monetary mã 11 (`T-1`).
- **Slide 12 (Tỷ trọng & Lift theo `event_type`):** Top 5 mã (`1, 46, 40, 49, 11`) và hệ số Lift mua hàng ở tháng `T-1`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "src/scripts"))


def run_script(rel_path: str, *args: str) -> None:
    """Run a repository script with the active Python interpreter."""
    script = ROOT / rel_path
    assert script.exists(), f"Missing script: {script}"
    cmd = [sys.executable, str(script), *args]
    print(f"$ {' '.join(cmd)}")
    subprocess.run(cmd, cwd=ROOT, check=True)


def show_chart(rel_path: str, caption: str | None = None, width: int = 920) -> None:
    """Display a generated chart image inline if present."""
    path = ROOT / rel_path
    if caption:
        display(Markdown(f"**{caption}** (`{rel_path}`)"))
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"_Chưa có file ảnh `{rel_path}` — chạy cell tạo biểu đồ phía trên._"))


def show_table(rel_path: str, nrows: int = 12) -> None:
    """Display a generated CSV table inline if present."""
    path = ROOT / rel_path
    if path.exists():
        display(pd.read_csv(path, nrows=nrows))
    else:
        display(Markdown(f"_Chưa có bảng `{rel_path}`._"))

## 1. Chất lượng dữ liệu TRX, Null theo cặp & Cơ cấu Currency (Slide 8)

In [ ]:
RUN_TRX_QUALITY = False
if RUN_TRX_QUALITY:
    run_script("src/scripts/audit_trx_nulls.py")
    run_script("src/audit_mbd_full_history_missingness.py")
    run_script("outputs/trx_charts/plot_currency.py")

show_table("outputs/eda_insights_20261119/amount_by_currency.csv", nrows=15)
show_chart("outputs/trx_charts/trx_currency.png", "Slide 8 · Cơ cấu giao dịch theo currency")

## 2. Phân vị số giao dịch & tổng Amount mã 11, Ngoại lai lớn nhất (Slide 9)

In [ ]:
RUN_TRX_QUANTILES = False
if RUN_TRX_QUANTILES:
    run_script("src/scripts/audit_trx_outliers.py")
    run_script("outputs/trx_charts/customer_quantiles.py")
    run_script("outputs/trx_charts/plot_customer_quantiles.py")

show_chart("outputs/trx_charts/customer_quantiles.png", "Slide 9 · Phân vị giao dịch & tổng amount mã 11 trên mỗi khách")

## 3. Đường cong Lorenz tập trung giao dịch theo khách (Slide 10)

In [ ]:
REGENERATE_LORENZ = False
if REGENERATE_LORENZ:
    run_script("outputs/trx_charts/plot_sparsity.py")

show_chart("outputs/trx_charts/sparsity_customer_concentration.png", "Slide 10 · Mức độ tập trung giao dịch theo khách")

## 4. RFM: Nhóm Mua và Không mua (Slide 11)

In [ ]:
REGENERATE_RFM = False
if REGENERATE_RFM:
    run_script("outputs/trx_charts/compute_rfm_means.py")
    run_script("outputs/trx_charts/plot_rfm.py")

show_table("outputs/trx_charts/rfm_means_p95.csv")
show_chart("outputs/trx_charts/trx_rfm_purchase_vs_no_purchase.png", "Slide 11 · RFM: Mua và Không mua")

## 5. Tỷ trọng & Hệ số Lift theo `event_type` tháng `T-1` (Slide 12)

In [ ]:
REGENERATE_EVENT_TYPE = False
if REGENERATE_EVENT_TYPE:
    run_script("outputs/trx_charts/generate_event_type_lift_data.py")
    run_script("outputs/trx_charts/plot_event_type_portrait.py")
    run_script("outputs/trx_charts/plot_event_type_lift_horizontal.py")

show_table("outputs/trx_charts/trx_top5_event_type_lift.csv")
show_chart("outputs/trx_charts/trx_top5_event_type_portrait.png", "Slide 12a · Tỷ trọng Top 5 event_type")
show_chart("outputs/trx_charts/trx_event_type_lift_horizontal.png", "Slide 12b · Lift theo event_type tháng T-1")